In [ ]:
import os 
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing_extensions import TypedDict
from IPython.display import display, Image

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

class State(TypedDict):
    topic: str
    story: str
    enhanced_story: str
    final_story: str

def generate_story(state:State):
    response = model.invoke(f"Write a one sentence story premise on topic {state['topic']}")
    return {"story": response.content}

def enhance_story(state:State):
    response = model.invoke(f"Enhance the story premise with valid details : {state['story']}")
    return {"enhanced_story": response.content}

def final_story(state:State):
    response = model.invoke(f"Add an unexpected twist to the story premise {state['enhanced_story']}")
    return {"final_story": response.content}

def check_conflict(state:State):
    if "?" in state["story"] or "!" in state["story"]:
        return "Fail"
    else:
        return "Pass"
    

# Graph creation
graph = StateGraph(State)

graph.add_node("Generate Story", generate_story)
graph.add_node("Enhance Story", enhance_story)
graph.add_node("Improve Story", final_story)

graph.add_edge(START, "Generate Story")
graph.add_conditional_edges("Generate Story", check_conflict, {"Fail": "Generate Story", "Pass": "Enhance Story"})
graph.add_edge("Enhance Story", "Improve Story")
graph.add_edge("Improve Story", END)

graph_builder = graph.compile()

display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
graph_builder.invoke({"topic": "rabbit and tortoise"})

## Parallization

In [ ]:
import os 
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing_extensions import TypedDict
from IPython.display import display, Image

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

class State(TypedDict):
  topic: str
  characters: str
  settings: str
  premise: str
  final_story: str

# Nodes
def generate_character(state:State):
  """Generate the characters of the story"""
  response = model.invoke(f"Create two character names and brief traits for a story about {state['topic']}")
  return {"characters": response.content}

def generate_settings(state:State):
  """Generate the settings of the story"""
  response = model.invoke(f"Generate a vivid settings for a story about {state['topic']}")
  return {"settings": response.content}

def generate_premise(state:State):
  """Generate the premise of the story"""
  response = model.invoke(f"Write a one sentence plot premise for a story about {state['topic']}")
  return {"premise": response.content}

def generate_story(state:State):
  """Generate the story"""
  response = model.invoke(f"""Write a story introduction using these elements.
  Characters: {state["characters"]}
  Settings: {state["settings"]}
  Premise: {state["premise"]}""")
  return {"final_story": response.content}


graph = StateGraph(State)

graph.add_node("Generate Characters", generate_character)
graph.add_node("Generate Settings", generate_settings)
graph.add_node("Generate Premise", generate_premise)
graph.add_node("Generate Story", generate_story)

graph.add_edge(START, "Generate Characters")
graph.add_edge(START, "Generate Settings")
graph.add_edge(START, "Generate Premise")
graph.add_edge("Generate Characters", "Generate Story")
graph.add_edge("Generate Settings", "Generate Story")
graph.add_edge("Generate Premise", "Generate Story")
graph.add_edge("Generate Story", END)

graph_builder = graph.compile()

display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
state = {"topic": "time travel"}
response = graph_builder.invoke(state)
response

## Router

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Literal
from pydantic import BaseModel, Field
from IPython.display import display, Image
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

# routing class
class Router(BaseModel):
  step: Literal["poem", "story", "joke"] = Field(description="The next step in the routing process")

router = model.with_structured_output(Router)

# graph state class
class State(TypedDict):
    input: str
    decision: str
    output: str

# graph nodes
def llm_call_1(state:State):
  """Write a story"""
  print("llm_call_1 is executing......")
  response = model.invoke(state['input'])
  return {"output": response.content}

def llm_call_2(state:State):
  """Write a joke"""
  print("llm_call_2 is executing......")
  response = model.invoke(state['input'])
  return {"output": response.content}

def llm_call_3(state:State):
  """Write a poem"""
  print("llm_call_3 is executing......")
  response = model.invoke(state['input'])
  return {"output": response.content}

# routing nodes
def llm_router_fn(state:State):
  """Route the input the appopriate node"""
  decision = router.invoke([
      SystemMessage(
          content="Route the input story, joke, or poem based on the user request"
      ), 
      HumanMessage(content=state['input'])
  ])
  return {"decision": decision.step}

# routing condition
def routing_decision(state:State):
  """Return the node name you want to visit next"""
  if state['decision'] == "story":
    return "llm_call_1"
  elif state['decision'] == "joke":
    return "llm_call_2"
  elif state['decision'] == "poem":
    return "llm_call_3"


# Graph creation
graph = StateGraph(State)

graph.add_node("llm_call_1", llm_call_1)
graph.add_node("llm_call_2", llm_call_2)
graph.add_node("llm_call_3", llm_call_3)
graph.add_node("llm_router_fn", llm_router_fn)

graph.add_edge(START, "llm_router_fn")
graph.add_conditional_edges(
    "llm_router_fn", 
    routing_decision, 
     {"llm_call_1": "llm_call_1", "llm_call_2": "llm_call_2", "llm_call_3": "llm_call_3"})
graph.add_edge("llm_call_1", END)
graph.add_edge("llm_call_2", END)
graph.add_edge("llm_call_3", END)

graph_builder = graph.compile()

display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
response = graph_builder.invoke({"input": "Tell me a joke on Agentic AI System"})
response

## Orchestration

In [ ]:
import os
import operator
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from typing_extensions import TypedDict
from typing import Literal
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.types import Send

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

class Section(BaseModel):
  name: str = Field(description="name of this section in the report")
  description: str = Field(description="Breif overview of the main topics and concepts of the section")

class Sections(BaseModel):
  sections: list[Section] = Field(description="list of sections in the report")

planner = model.with_structured_output(Sections)

class State(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str

class Worker(TypedDict):
  section: Section

def orchestration(state:State):
  """Orchestration that generates a plan for the report"""
  report_sections = planner.invoke(
      [
          SystemMessage(content="Generate a plan for the report"),
          HumanMessage(content=f"Generate a plan for a report on the topic {state['topic']}")
      ]
  )

  print("Report Section generated......")
  return {"sections": report_sections.sections}

def llm_call(state:Worker):
  """Worker writes the section of the report"""

  sections = model.invoke(
      [
          SystemMessage(
              content = "Write a report section following the provided name and description. Include no premeble for each section. Use Markdown formatting"
          ),
          HumanMessage(
              content=f"Here is the section name: {state['section'].name} and description: {state['section'].description}"
          )
      ]
  )

  print("Complete section generated......")
  return {"completed_sections": [sections.content]}

def assign_worker(state:State):
  """Assign a worker to each sections in the plan"""
  return [Send("llm_call", {"section": s}) for s in state["sections"]]

def synthesizer(state:State):
  """Synthesize the full report from sections"""
  completed = state["completed_sections"]
  return {"final_report": "\n\n--\n\n".join(completed)}

# Graph
graph = StateGraph(State)

graph.add_node("Orchestration", orchestration)
graph.add_node("llm_call", llm_call)
graph.add_node("synthesizer", synthesizer)

graph.add_edge(START, "Orchestration")
graph.add_conditional_edges(
    "Orchestration",
    assign_worker,
    ["llm_call"]
)
graph.add_edge("llm_call", "synthesizer")
graph.add_edge("synthesizer", END)

compiled_graph = graph.compile()

display(Image(compiled_graph.get_graph().draw_mermaid_png()))

# Invokation
response = compiled_graph.invoke({"topic": "Create a report on Agentic AI RAGs"})

Markdown(response["final_report"])

In [ ]:
section_list = planner.invoke(
      [
          SystemMessage(content="Generate a plan for the report"),
          HumanMessage(content=f"Generate a plan for a report on the topic Agentic AI")
      ]
  ).sections

for section in section_list:
  print(section.name, "------>", section.description)

In [ ]:
responses = []
for section in section_list:
  response = model.invoke(
                  [
                      SystemMessage(
                          content = "Write a report section following the provided name and description. Include no premeble for each section. Use Markdown formatting"
                      ),
                      HumanMessage(
                          content=f"Here is the section name: {section.name} and description: {section.description}"
                      )
                  ]
              )
  responses.append({
      "section_name": section.name,
      "section_description": section.description,
      "output": response.content
  })

for response in responses:
   print(response, end="\n\n")

print(responses[0]["output"])